# Módulo 53: Modelos de regresión — Solución
Author: Juliho Castillo Colmenares Ph.D.

Enunciado para estudiantes: [m53-modelos-de-regresion-guia.ipynb](m53-modelos-de-regresion-guia.ipynb) · Lección: [m53-modelos-de-regresion-leccion.ipynb](m53-modelos-de-regresion-leccion.ipynb)

## Contexto

Una empresa vende un producto en 200 mercados y reparte su presupuesto de publicidad entre televisión, radio y periódico. Marketing quiere saber qué medios influyen en las ventas y, para planear la próxima campaña, un **rango** de ventas esperadas al 90 % de confianza para un presupuesto concreto, después de comprobar que el modelo cumple los supuestos de mínimos cuadrados ordinarios.

## Datos

`advertising.csv` (en esta carpeta): 200 mercados con el presupuesto en `TV`, `Radio` y `Newspaper` (miles de dólares) y las ventas `Sales` (miles de unidades).

## Instrucciones

1. **Preparar los datos:** variables `TV`, `Radio`, `Newspaper` y objetivo `Sales`; partición 70/30 con `random_state=1`.
2. **Eliminación hacia atrás:** quitar la variable con mayor p-valor mientras sea mayor que 0.05.
3. **Evaluar el modelo final:** $R^2$ y $R^2$ ajustado de entrenamiento; MSE y MAE de prueba.
4. **Pronóstico de intervalo al 90 %** para TV = 100, Radio = 50 y Newspaper = 70: pronóstico puntual, intervalo de confianza de la media e intervalo de predicción de una observación nueva.
5. **Validar los supuestos:** Jarque–Bera, White, Durbin–Watson y VIF (con constante).

*El enunciado original del curso no se conserva; estas instrucciones se reconstruyeron a partir de la solución.* El objetivo (pronóstico de intervalo y validación de supuestos) y la fórmula del intervalo provienen de la guía de clase del módulo.


### Tarea 1. Preparar los datos

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_white
from statsmodels.stats.stattools import jarque_bera, durbin_watson
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
# Load the data
data = pd.read_csv("advertising.csv")
data.head()

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,12.0
3,151.5,41.3,58.5,16.5
4,180.8,10.8,58.4,17.9


In [3]:
# Define features and target variable
X = data[["TV", "Radio", "Newspaper"]]
y = data["Sales"]

In [4]:
# Split data into training (70%) and testing (30%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

### Tarea 2. Eliminación hacia atrás

In [5]:
# Add a constant to the model (intercept)
X_train_const = sm.add_constant(X_train)

In [6]:
# Fit the initial model
model = sm.OLS(y_train, X_train_const).fit()

In [7]:
# Stepwise selection process
significant_features = list(X.columns)
significant_features

['TV', 'Radio', 'Newspaper']

In [8]:
while True:
    model = sm.OLS(y_train, sm.add_constant(X_train[significant_features])).fit()
    p_values = model.pvalues.iloc[1:]  # Exclude intercept
    max_p_value = p_values.max()
    if max_p_value > 0.05:
        excluded_feature = p_values.idxmax()
        significant_features.remove(excluded_feature)
    else:
        break

In [9]:
# Final model with significant features only
X_train_final = X_train[significant_features]
X_train_final_const = sm.add_constant(X_train_final)
final_model = sm.OLS(y_train, X_train_final_const).fit()

### Tarea 3. Evaluar el modelo final

In [10]:
# Evaluate model on test data
X_test_final = X_test[significant_features]
X_test_final_const = sm.add_constant(X_test_final)
y_pred = final_model.predict(X_test_final_const)

In [11]:
# Calculate R-squared, Adjusted R-squared, MSE, and MAE
r_squared = final_model.rsquared
adjusted_r_squared = final_model.rsquared_adj
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R-squared: {r_squared}")
print(f"Adjusted R-squared: {adjusted_r_squared}")
print(f"MSE: {mse}")
print(f"MAE: {mae}")

R-squared: 0.8993439479032659
Adjusted R-squared: 0.8978745164857953
MSE: 2.3645069433762367
MAE: 1.1919753277836762


In [12]:
X_train_final_const

,const,TV,Radio
116,1.0,139.2,14.3
67,1.0,139.3,14.5
78,1.0,5.4,29.9
42,1.0,293.6,27.7
17,1.0,281.4,39.6
...,...,...,...
133,1.0,219.8,33.5
137,1.0,273.7,28.9
72,1.0,26.8,33.0
140,1.0,73.4,17.0


In [13]:
# Confirm the significant features in the final model
print("Significant features in the final model:", significant_features)

Significant features in the final model: ['TV', 'Radio']


### Tarea 4. Pronóstico de intervalo al 90 %

In [14]:
# Prepare new data based on only the significant features

# Creating the new input values dictionary with only significant features
new_data_values = {"TV": 100, "Radio": 50, "Newspaper": 70}
# Filter the dictionary to include only the significant features
new_data_final = pd.DataFrame(
    [{feature: new_data_values[feature] for feature in significant_features}]
)

# Add a constant to align with the model's intercept term
new_data_final_const = sm.add_constant(new_data_final, has_constant="add")
new_data_final_const

,const,TV,Radio
0,1.0,100,50


In [15]:
# Predict with 90% intervals for the filtered input
predicted_sales = final_model.get_prediction(new_data_final_const)
frame_90 = predicted_sales.summary_frame(alpha=0.1)  # 90% level

# Confidence interval: uncertainty about the MEAN sales for these budgets
ci_90 = predicted_sales.conf_int(alpha=0.1)
# Prediction interval: range for ONE new observation (adds the residual variance, so it is wider)
pi_90 = frame_90[["obs_ci_lower", "obs_ci_upper"]].values

print(f"Point prediction: {frame_90['mean'].iloc[0]}")
print(f"90% Confidence Interval for the mean sales: {ci_90}")
print(f"90% Prediction Interval for a new observation: {pi_90}")

Point prediction: 15.221675664877203
90% Confidence Interval for the mean sales: [[14.71820447 15.72514686]]
90% Prediction Interval for a new observation: [[12.34281323 18.1005381 ]]


### Tarea 5. Validar los supuestos

In [16]:
# Assumption Validation
# Residuals
residuals = final_model.resid

In [17]:
# 1. Normality of Residuals (Jarque-Bera)
jb_stat, jb_pvalue, _, __ = jarque_bera(residuals)
print(f"Jarque-Bera Test: Statistic={jb_stat}, p-value={jb_pvalue}")

Jarque-Bera Test: Statistic=24.662861496030988, p-value=4.410904674141444e-06


In [18]:
# 2. Homoscedasticity (White's Test)
white_test = het_white(residuals, final_model.model.exog)
print(f"White's Test: Statistic={white_test[0]}, p-value={white_test[1]}")

White's Test: Statistic=11.353378151633706, p-value=0.04480664163871973


In [19]:
# 3. Autocorrelation (Durbin-Watson Test)
dw_stat = durbin_watson(residuals)
print(f"Durbin-Watson Statistic: {dw_stat}")

Durbin-Watson Statistic: 2.0375161916374998


In [20]:
# 4. Multicollinearity (Variance Inflation Factor - VIF)
# The VIF is computed on the design matrix with the constant; the constant's row is not reported
X_train_vif = sm.add_constant(X_train_final)
vif_data = pd.DataFrame()
vif_data["feature"] = X_train_vif.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_train_vif.values, i)
    for i in range(X_train_vif.shape[1])
]
vif_data = vif_data[vif_data["feature"] != "const"].reset_index(drop=True)
print(vif_data)

  feature       VIF
0      TV  1.001352
1   Radio  1.001352


In [21]:
# Interpret Results
if jb_pvalue > 0.1:
    print("The residuals are normally distributed (Jarque-Bera test).")
else:
    print("The residuals do not appear normally distributed (Jarque-Bera test).")

if white_test[1] > 0.1:
    print("Homoscedasticity is present (White's test).")
else:
    print("Heteroscedasticity detected (White's test).")

if 1.5 < dw_stat < 2.5:
    print("No significant autocorrelation detected (Durbin-Watson test).")
else:
    print("Autocorrelation may be present (Durbin-Watson test).")

if vif_data["VIF"].max() < 10:
    print("No multicollinearity issues (VIF test).")
else:
    print("Multicollinearity detected (VIF test).")

The residuals do not appear normally distributed (Jarque-Bera test).
Heteroscedasticity detected (White's test).
No significant autocorrelation detected (Durbin-Watson test).
No multicollinearity issues (VIF test).


**Revisión posterior (2026-10):** el intervalo al 90 % que se reportaba como de la predicción, [14.72, 15.73], es el intervalo de confianza de la **media**; se añadió el intervalo de **predicción** para una observación nueva, [12.34, 18.10], con `summary_frame(alpha=0.1)` (columnas `obs_ci_*`). El VIF ahora se calcula con constante (1.00 para TV y Radio, antes 2.11 sin constante). El PDF conserva la versión entregada.